# 75. The VRP with Pickup and Delivery (VRPPD)
## Tier 3: The Advanced Algorithm (Genetic Algorithm)

### Key assumptions
- Population-based evolution with genetic operators
- Two-part chromosome encoding: route assignment + visit sequence
- Order Crossover (OX) adapted for pickup-delivery pairs
- Fitness function includes penalty for constraint violations
- Elitism preserves best solutions across generations

### Approach (step-by-step)
1. **Chromosome Encoding**: Two-part representation for routes and sequences
2. **Population Initialization**: Generate diverse feasible initial solutions
3. **Fitness Evaluation**: Calculate total distance plus constraint penalties
4. **Selection**: Tournament selection for parent choosing
5. **Crossover**: Order Crossover maintaining precedence constraints
6. **Mutation**: Intra-route and inter-route moves preserving feasibility
7. **Replacement**: Generational replacement with elitism

### What to look for in the results
- Convergence behavior over generations
- Population diversity and exploration vs exploitation balance
- Solution quality improvement over initial population
- Constraint satisfaction rate (100% for feasible solutions)

### Concrete example (from the source)
10-pair VRPPD instance with 3 vehicles:
- Population size: 100, Generations: 500
- Expected best solution: 3 routes with total distance 156.7 units
- Expected Route 1: [0,2,7,4,9,0] serving pairs (2,7) and (4,9)
- Expected Route 2: [0,1,6,3,8,5,10,0] serving pairs (1,6), (3,8), and (5,10)
- Expected improvement: 23% over initial solution

In [1]:
# Import required libraries for Genetic Algorithm
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dataclasses import dataclass
from typing import List, Tuple, Dict, Optional
import random
import copy
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully for VRPPD Genetic Algorithm")

In [ ]:
@dataclass
class VRPPDInstance:
    """Data structure for VRPPD problem instance"""
    num_pairs: int
    num_vehicles: int
    vehicle_capacity: int
    distances: np.ndarray
    demands: List[int]
    
    def __post_init__(self):
        self.num_vertices = 2 * self.num_pairs + 2
        self.depot_start = 0
        self.depot_end = 2 * self.num_pairs + 1
        self.pickups = list(range(1, self.num_pairs + 1))
        self.deliveries = list(range(self.num_pairs + 1, 2 * self.num_pairs + 1))
        self.all_vertices = list(range(self.num_vertices))
        self.vehicles = list(range(self.num_vehicles))

@dataclass
class Chromosome:
    """Represents a solution chromosome for VRPPD"""
    route_assignment: List[int]  # Which vehicle serves each pair
    sequences: List[List[int]]   # Visit sequence for each vehicle
    fitness: float
    feasible: bool
    
    def copy(self):
        return Chromosome(
            self.route_assignment.copy(),
            [seq.copy() for seq in self.sequences],
            self.fitness,
            self.feasible
        )

# Create a larger instance for GA demonstration
def create_ga_instance():
    """Create a 10-pair instance suitable for Genetic Algorithm"""
    num_pairs = 10
    num_vehicles = 3
    vehicle_capacity = 15
    
    # Generate random demands for 10 pairs
    np.random.seed(42)  # For reproducibility
    pickup_demands = np.random.randint(1, 6, num_pairs).tolist()
    delivery_demands = [-d for d in pickup_demands]
    demands = pickup_demands + delivery_demands
    
    # Generate distance matrix for 22 vertices (depot + 10 pickups + 10 deliveries + depot_end)
    num_vertices = 2 * num_pairs + 2
    distances = np.random.rand(num_vertices, num_vertices) * 30 + 5
    
    # Make symmetric and set diagonal to 0
    distances = (distances + distances.T) / 2
    np.fill_diagonal(distances, 0)
    
    # Ensure depot distances are reasonable
    distances[0, :] = np.random.rand(num_vertices) * 20 + 5
    distances[:, 0] = distances[0, :]
    distances[0, 0] = 0
    distances[-1, :] = distances[0, :]
    distances[:, -1] = distances[:, 0, :]
    
    return VRPPDInstance(num_pairs, num_vehicles, vehicle_capacity, distances, demands)

# Create the instance
ga_instance = create_ga_instance()
print(f"VRPPD GA Instance created:")
print(f"- Pickup-delivery pairs: {ga_instance.num_pairs}")
print(f"- Vehicles: {ga_instance.num_vehicles}")
print(f"- Vehicle capacity: {ga_instance.vehicle_capacity}")
print(f"- Total vertices: {ga_instance.num_vertices}")
print(f"- Demands: {ga_instance.demands}")

In [ ]:
def calculate_route_distance(route: List[int], distances: np.ndarray) -> float:
    """Calculate total distance of a route"""
    if len(route) < 2:
        return 0.0
    total_distance = 0.0
    for i in range(len(route) - 1):
        total_distance += distances[route[i]][route[i + 1]]
    return total_distance

def check_precedence_constraint(sequence: List[int], pair_idx: int, num_pairs: int) -> bool:
    """Check if pickup comes before delivery in sequence"""
    pickup = pair_idx
    delivery = pair_idx + num_pairs
    
    try:
        pickup_idx = sequence.index(pickup)
        delivery_idx = sequence.index(delivery)
        return pickup_idx < delivery_idx
    except ValueError:
        return True  # If either is not in sequence, constraint is satisfied

def check_capacity_constraint(sequence: List[int], demands: List[int], capacity: int) -> bool:
    """Check if route respects capacity constraints"""
    current_load = 0
    
    for vertex in sequence:
        if vertex == 0 or vertex == len(demands) + 1:  # Depot vertices
            continue
        
        demand = demands[vertex - 1]
        current_load += demand
        
        if current_load > capacity or current_load < 0:
            return False
    
    return True

def evaluate_chromosome(chromosome: Chromosome, instance: VRPPDInstance) -> Tuple[float, bool]:
    """Evaluate chromosome fitness and feasibility
    
    Returns: (fitness_value, is_feasible)
    """
    total_distance = 0.0
    penalty = 0.0
    feasible = True
    
    # Build routes from chromosome
    routes = []
    for vehicle_idx in range(instance.num_vehicles):
        route = [instance.depot_start]
        
        # Add assigned pairs in sequence order
        if vehicle_idx < len(chromosome.sequences):
            for pair_idx in chromosome.sequences[vehicle_idx]:
                if chromosome.route_assignment[pair_idx - 1] == vehicle_idx:
                    route.append(pair_idx)  # Pickup
                    route.append(pair_idx + instance.num_pairs)  # Delivery
        
        route.append(instance.depot_end)
        routes.append(route)
    
    # Calculate total distance and check constraints
    for route in routes:
        if len(route) > 2:  # Route serves at least one pair
            # Check capacity constraint
            if not check_capacity_constraint(route, instance.demands, instance.vehicle_capacity):
                penalty += 1000  # Large penalty for capacity violation
                feasible = False
            
            # Check precedence constraints
            for pair_idx in range(1, instance.num_pairs + 1):
                if pair_idx in route and (pair_idx + instance.num_pairs) in route:
                    if not check_precedence_constraint(route, pair_idx, instance.num_pairs):
                        penalty += 500  # Penalty for precedence violation
                        feasible = False
            
            total_distance += calculate_route_distance(route, instance.distances)
    
    # Fitness = total distance + penalty (lower is better)
    fitness = total_distance + penalty
    
    return fitness, feasible

print("Chromosome evaluation functions defined")

In [ ]:
def generate_initial_population(instance: VRPPDInstance, population_size: int) -> List[Chromosome]:
    """Generate initial population with diverse feasible solutions"""
    population = []
    
    for _ in range(population_size):
        # Random route assignment
        route_assignment = [random.randint(0, instance.num_vehicles - 1) 
                           for _ in range(instance.num_pairs)]
        
        # Build sequences for each vehicle
        sequences = [[] for _ in range(instance.num_vehicles)]
        
        for pair_idx in range(1, instance.num_pairs + 1):
            vehicle = route_assignment[pair_idx - 1]
            sequences[vehicle].append(pair_idx)
        
        # Randomize sequences within each vehicle
        for vehicle_seq in sequences:
            random.shuffle(vehicle_seq)
        
        # Create chromosome
        chromosome = Chromosome(route_assignment, sequences, 0.0, False)
        
        # Evaluate fitness
        fitness, feasible = evaluate_chromosome(chromosome, instance)
        chromosome.fitness = fitness
        chromosome.feasible = feasible
        
        population.append(chromosome)
    
    return population

def tournament_selection(population: List[Chromosome], tournament_size: int = 3) -> Chromosome:
    """Select parent using tournament selection"""
    tournament = random.sample(population, min(tournament_size, len(population)))
    return min(tournament, key=lambda x: x.fitness)

def order_crossover(parent1: Chromosome, parent2: Chromosome, instance: VRPPDInstance) -> Tuple[Chromosome, Chromosome]:
    """Order Crossover (OX) adapted for VRPPD pickup-delivery pairs"""
    # Flatten sequences for crossover
    seq1 = []
    for vehicle_seq in parent1.sequences:
        seq1.extend(vehicle_seq)
    
    seq2 = []
    for vehicle_seq in parent2.sequences:
        seq2.extend(vehicle_seq)
    
    if len(seq1) < 2 or len(seq2) < 2:
        return parent1.copy(), parent2.copy()
    
    # Select crossover points
    start = random.randint(0, len(seq1) - 1)
    end = random.randint(start + 1, len(seq1))
    
    # Create offspring
    offspring1_seq = [-1] * len(seq1)
    offspring2_seq = [-1] * len(seq2)
    
    # Copy middle segment
    offspring1_seq[start:end] = seq1[start:end]
    offspring2_seq[start:end] = seq2[start:end]
    
    # Fill remaining positions
    def fill_remaining(offspring, source, target):
        target_idx = 0
        for i in range(len(offspring)):
            if offspring[i] == -1:
                while target_idx < len(source) and source[target_idx] in offspring:
                    target_idx += 1
                if target_idx < len(source):
                    offspring[i] = source[target_idx]
                    target_idx += 1
        return offspring
    
    offspring1_seq = fill_remaining(offspring1_seq, seq2, seq1)
    offspring2_seq = fill_remaining(offspring2_seq, seq1, seq2)
    
    # Rebuild chromosome structures
    def rebuild_chromosome(sequence, parent):
        # Create new route assignment based on parent
        route_assignment = parent.route_assignment.copy()
        
        # Rebuild sequences
        sequences = [[] for _ in range(instance.num_vehicles)]
        
        for pair_idx in sequence:
            vehicle = route_assignment[pair_idx - 1]
            sequences[vehicle].append(pair_idx)
        
        return Chromosome(route_assignment, sequences, 0.0, False)
    
    offspring1 = rebuild_chromosome(offspring1_seq, parent1)
    offspring2 = rebuild_chromosome(offspring2_seq, parent2)
    
    return offspring1, offspring2

print("Genetic operators defined")

In [ ]:
def mutate_chromosome(chromosome: Chromosome, instance: VRPPDInstance, mutation_rate: float = 0.1) -> Chromosome:
    """Apply mutation operations to chromosome"""
    mutated = chromosome.copy()
    
    if random.random() < mutation_rate:
        # Intra-route mutation: swap two pairs in the same vehicle sequence
        for vehicle_idx in range(instance.num_vehicles):
            if len(mutated.sequences[vehicle_idx]) >= 2:
                if random.random() < mutation_rate:
                    seq = mutated.sequences[vehicle_idx]
                    i, j = random.sample(range(len(seq)), 2)
                    seq[i], seq[j] = seq[j], seq[i]
    
    if random.random() < mutation_rate:
        # Inter-route mutation: move a pair to another vehicle
        for pair_idx in range(1, instance.num_pairs + 1):
            if random.random() < mutation_rate / instance.num_pairs:
                current_vehicle = mutated.route_assignment[pair_idx - 1]
                new_vehicle = random.randint(0, instance.num_vehicles - 1)
                
                if new_vehicle != current_vehicle:
                    # Remove from old vehicle sequence
                    if pair_idx in mutated.sequences[current_vehicle]:
                        mutated.sequences[current_vehicle].remove(pair_idx)
                    
                    # Add to new vehicle sequence
                    mutated.sequences[new_vehicle].append(pair_idx)
                    mutated.route_assignment[pair_idx - 1] = new_vehicle
    
    # Re-evaluate fitness
    fitness, feasible = evaluate_chromosome(mutated, instance)
    mutated.fitness = fitness
    mutated.feasible = feasible
    
    return mutated

def genetic_algorithm(instance: VRPPDInstance, population_size: int = 100, 
                   generations: int = 500, crossover_rate: float = 0.8,
                   mutation_rate: float = 0.1, elitism_size: int = 2) -> Tuple[Chromosome, List[float]]:
    """Genetic Algorithm for VRPPD
    
    Returns: (best_chromosome, fitness_history)
    """
    print(f"Running Genetic Algorithm for VRPPD...")
    print(f"Population size: {population_size}, Generations: {generations}")
    
    # Initialize population
    population = generate_initial_population(instance, population_size)
    
    # Track best fitness over generations
    fitness_history = []
    best_chromosome = min(population, key=lambda x: x.fitness)
    
    print(f"Initial best fitness: {best_chromosome.fitness:.2f}")
    
    for generation in range(generations):
        # Elitism: preserve best individuals
        new_population = sorted(population, key=lambda x: x.fitness)[:elitism_size]
        
        # Generate offspring
        while len(new_population) < population_size:
            # Selection
            parent1 = tournament_selection(population)
            parent2 = tournament_selection(population)
            
            # Crossover
            if random.random() < crossover_rate:
                offspring1, offspring2 = order_crossover(parent1, parent2, instance)
            else:
                offspring1, offspring2 = parent1.copy(), parent2.copy()
            
            # Mutation
            offspring1 = mutate_chromosome(offspring1, instance, mutation_rate)
            offspring2 = mutate_chromosome(offspring2, instance, mutation_rate)
            
            # Add to new population
            new_population.append(offspring1)
            if len(new_population) < population_size:
                new_population.append(offspring2)
        
        # Replace population
        population = new_population
        
        # Update best solution
        current_best = min(population, key=lambda x: x.fitness)
        if current_best.fitness < best_chromosome.fitness:
            best_chromosome = current_best
        
        # Record fitness
        avg_fitness = sum(ind.fitness for ind in population) / len(population)
        fitness_history.append(best_chromosome.fitness)
        
        # Progress reporting
        if generation % 50 == 0:
            feasible_count = sum(1 for ind in population if ind.feasible)
            print(f"Generation {generation:3d}: Best = {best_chromosome.fitness:.2f}, "
                  f"Avg = {avg_fitness:.2f}, Feasible = {feasible_count}/{population_size}")
    
    print(f"\nGenetic Algorithm completed:")
    print(f"Final best fitness: {best_chromosome.fitness:.2f}")
    print(f"Feasibility: {'Yes' if best_chromosome.feasible else 'No'}")
    
    return best_chromosome, fitness_history

print("Genetic Algorithm implementation complete")

In [ ]:
# Run the Genetic Algorithm
best_chromosome, fitness_history = genetic_algorithm(
    ga_instance, 
    population_size=100, 
    generations=500, 
    crossover_rate=0.8, 
    mutation_rate=0.1, 
    elitism_size=2
)

In [ ]:
def extract_routes_from_chromosome(chromosome: Chromosome, instance: VRPPDInstance) -> List[Dict]:
    """Extract detailed route information from chromosome"""
    routes = []
    
    for vehicle_idx in range(instance.num_vehicles):
        route = [instance.depot_start]
        pairs_served = []
        total_demand = 0
        
        # Add assigned pairs in sequence order
        if vehicle_idx < len(chromosome.sequences):
            for pair_idx in chromosome.sequences[vehicle_idx]:
                if chromosome.route_assignment[pair_idx - 1] == vehicle_idx:
                    route.append(pair_idx)  # Pickup
                    route.append(pair_idx + instance.num_pairs)  # Delivery
                    pairs_served.append(pair_idx)
                    total_demand += instance.demands[pair_idx - 1]  # Pickup demand
        
        route.append(instance.depot_end)
        
        if len(pairs_served) > 0:  # Only include routes that serve pairs
            distance = calculate_route_distance(route, instance.distances)
            
            routes.append({
                'vehicle': vehicle_idx + 1,
                'route': route,
                'pairs_served': pairs_served,
                'total_demand': total_demand,
                'distance': distance
            })
    
    return routes

# Extract and analyze the best solution
best_routes = extract_routes_from_chromosome(best_chromosome, ga_instance)

print("\n=== BEST SOLUTION ANALYSIS ===")
print(f"Total fitness (distance + penalties): {best_chromosome.fitness:.2f}")
print(f"Solution feasible: {'Yes' if best_chromosome.feasible else 'No'}")
print(f"Number of routes used: {len(best_routes)}")

total_distance = sum(route['distance'] for route in best_routes)
print(f"Total distance: {total_distance:.2f}")

print("\nDetailed routes:")
for route_info in best_routes:
    route_str = " → ".join(["D" if v == 0 else 
                           f"P{v}" if v in ga_instance.pickups else 
                           f"D{v-ga_instance.num_pairs}" if v in ga_instance.deliveries else "D"
                           for v in route_info['route']])
    print(f"\nVehicle {route_info['vehicle']}:")
    print(f"  Route: {route_str}")
    print(f"  Pairs served: {route_info['pairs_served']}")
    print(f"  Total demand: {route_info['total_demand']}")
    print(f"  Distance: {route_info['distance']:.2f}")
    
    # Check precedence constraints
    precedence_ok = True
    for pair_idx in route_info['pairs_served']:
        pickup = pair_idx
        delivery = pair_idx + ga_instance.num_pairs
        if pickup in route_info['route'] and delivery in route_info['route']:
            pickup_idx = route_info['route'].index(pickup)
            delivery_idx = route_info['route'].index(delivery)
            if pickup_idx >= delivery_idx:
                precedence_ok = False
                break
    
    print(f"  Precedence constraints: {'✓ Satisfied' if precedence_ok else '✗ Violated'}")

In [ ]:
def visualize_ga_solution(instance: VRPPDInstance, best_chromosome: Chromosome, 
                         fitness_history: List[float], routes: List[Dict]):
    """Create comprehensive visualization of the GA solution"""
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('VRPPD Genetic Algorithm - Solution Analysis', fontsize=16, fontweight='bold')
    
    # 1. Convergence Plot
    ax1.set_title('Fitness Convergence Over Generations', fontweight='bold')
    ax1.plot(fitness_history, 'b-', linewidth=2, label='Best Fitness')
    ax1.set_xlabel('Generation')
    ax1.set_ylabel('Fitness (Distance + Penalties)')
    ax1.grid(True, alpha=0.3)
    ax1.legend()
    
    # Add convergence statistics
    initial_fitness = fitness_history[0]
    final_fitness = fitness_history[-1]
    improvement = ((initial_fitness - final_fitness) / initial_fitness) * 100
    
    ax1.annotate(f'Initial: {initial_fitness:.1f}\nFinal: {final_fitness:.1f}\nImprovement: {improvement:.1f}%',
                xy=(0.05, 0.95), xycoords='axes fraction',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
                fontsize=10, verticalalignment='top')
    
    # 2. Route Network Visualization
    ax2.set_title('Best Solution Route Network', fontweight='bold')
    
    # Create positions for visualization (circular layout)
    num_vertices = instance.num_vertices
    angles = np.linspace(0, 2*np.pi, num_vertices, endpoint=False)
    positions = {}
    
    for i, angle in enumerate(angles):
        if i == 0:  # Depot
            positions[i] = (0, 0)
        elif i == num_vertices - 1:  # Depot end
            positions[i] = (0.5, 0.5)
        else:
            positions[i] = (np.cos(angle) * 3, np.sin(angle) * 3)
    
    colors = ['blue', 'red', 'green', 'orange', 'purple']
    
    # Draw routes
    for i, route_info in enumerate(routes):
        route_vertices = route_info['route']
        color = colors[i % len(colors)]
        
        # Draw route edges
        for j in range(len(route_vertices) - 1):
            start_pos = positions[route_vertices[j]]
            end_pos = positions[route_vertices[j + 1]]
            ax2.annotate('', xy=end_pos, xytext=start_pos,
                        arrowprops=dict(arrowstyle='->', color=color, lw=2, alpha=0.7))
    
    # Draw vertices
    for vertex, pos in positions.items():
        if vertex == 0 or vertex == num_vertices - 1:
            ax2.plot(pos[0], pos[1], 'ks', markersize=15, label='Depot')
            ax2.text(pos[0], pos[1] + 0.3, 'D', ha='center', fontweight='bold')
        elif vertex in instance.pickups:
            ax2.plot(pos[0], pos[1], 'o', color='lightgreen', markersize=8)
            ax2.text(pos[0], pos[1] + 0.2, f'P{vertex}', ha='center', fontsize=8)
        elif vertex in instance.deliveries:
            ax2.plot(pos[0], pos[1], 's', color='lightcoral', markersize=8)
            ax2.text(pos[0], pos[1] + 0.2, f'D{vertex - instance.num_pairs}', ha='center', fontsize=8)
    
    ax2.set_xlabel('X Coordinate')
    ax2.set_ylabel('Y Coordinate')
    ax2.grid(True, alpha=0.3)
    ax2.set_aspect('equal')
    
    # 3. Route Statistics
    ax3.set_title('Route Statistics', fontweight='bold')
    
    route_numbers = [f'Vehicle {r["vehicle"]}' for r in routes]
    route_distances = [r['distance'] for r in routes]
    route_demands = [r['total_demand'] for r in routes]
    route_pairs = [len(r['pairs_served']) for r in routes]
    
    x = np.arange(len(route_numbers))
    width = 0.25
    
    ax3_twin = ax3.twinx()
    
    bars1 = ax3.bar(x - width, route_distances, width, label='Distance', color='skyblue')
    bars2 = ax3.bar(x, route_demands, width, label='Demand', color='lightgreen')
    bars3 = ax3_twin.bar(x + width, route_pairs, width, label='Pairs', color='lightcoral')
    
    ax3.set_xlabel('Vehicles')
    ax3.set_ylabel('Distance / Demand')
    ax3_twin.set_ylabel('Number of Pairs')
    ax3.set_xticks(x)
    ax3.set_xticklabels(route_numbers)
    ax3.legend(loc='upper left')
    ax3_twin.legend(loc='upper right')
    ax3.grid(True, alpha=0.3)
    
    # 4. Performance Metrics
    ax4.set_title('Algorithm Performance Metrics', fontweight='bold')
    
    metrics = ['Total Distance', 'Routes Used', 'Feasibility', 'Improvement (%)']
    values = [
        sum(r['distance'] for r in routes),
        len(routes),
        1.0 if best_chromosome.feasible else 0.0,
        improvement
    ]
    
    bars = ax4.barh(metrics, values, color='steelblue')
    ax4.set_xlabel('Value')
    ax4.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, value in zip(bars, values):
        if isinstance(value, float):
            ax4.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height()/2,
                    f'{value:.2f}', ha='left', va='center', fontweight='bold')
        else:
            ax4.text(bar.get_width() + max(values) * 0.01, bar.get_y() + bar.get_height()/2,
                    f'{value}', ha='left', va='center', fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return fig

# Visualize the GA solution
fig = visualize_ga_solution(ga_instance, best_chromosome, fitness_history, best_routes)
print("\n=== SOLUTION VISUALIZATION ===")
print("Comprehensive analysis plots generated above.")

In [ ]:
def parameter_sensitivity_analysis():
    """Analyze GA performance with different parameter settings"""
    
    print("\n=== PARAMETER SENSITIVITY ANALYSIS ===")
    
    # Test different parameter combinations
    parameter_sets = [
        {'pop_size': 50, 'generations': 200, 'cross_rate': 0.7, 'mut_rate': 0.05},
        {'pop_size': 100, 'generations': 500, 'cross_rate': 0.8, 'mut_rate': 0.1},  # Base case
        {'pop_size': 150, 'generations': 500, 'cross_rate': 0.8, 'mut_rate': 0.1},
        {'pop_size': 100, 'generations': 300, 'cross_rate': 0.8, 'mut_rate': 0.1},
        {'pop_size': 100, 'generations': 500, 'cross_rate': 0.9, 'mut_rate': 0.1},
        {'pop_size': 100, 'generations': 500, 'cross_rate': 0.8, 'mut_rate': 0.15},
    ]
    
    results = []
    
    for i, params in enumerate(parameter_sets):
        print(f"\nTesting parameter set {i+1}: {params}")
        
        # Run GA with current parameters
        best_chromo, fitness_hist = genetic_algorithm(
            ga_instance,
            population_size=params['pop_size'],
            generations=params['generations'],
            crossover_rate=params['cross_rate'],
            mutation_rate=params['mut_rate'],
            elitism_size=2
        )
        
        # Calculate metrics
        final_fitness = fitness_hist[-1]
        initial_fitness = fitness_hist[0]
        improvement = ((initial_fitness - final_fitness) / initial_fitness) * 100
        convergence_gen = next((i for i, f in enumerate(fitness_hist) 
                              if f <= final_fitness * 1.01), len(fitness_hist))
        
        results.append({
            'params': params,
            'final_fitness': final_fitness,
            'improvement': improvement,
            'convergence_gen': convergence_gen,
            'feasible': best_chromo.feasible
        })
        
        print(f"  Final fitness: {final_fitness:.2f}")
        print(f"  Improvement: {improvement:.1f}%")
        print(f"  Convergence generation: {convergence_gen}")
        print(f"  Feasible: {'Yes' if best_chromo.feasible else 'No'}")
    
    # Create comparison visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle('GA Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
    
    labels = [f"Set {i+1}" for i in range(len(results))]
    
    # 1. Final Fitness Comparison
    final_fitnesses = [r['final_fitness'] for r in results]
    bars1 = ax1.bar(labels, final_fitnesses, color='skyblue')
    ax1.set_ylabel('Final Fitness')
    ax1.set_title('Final Fitness Comparison')
    ax1.grid(True, alpha=0.3)
    
    # Add value labels
    for bar, fitness in zip(bars1, final_fitnesses):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(final_fitnesses) * 0.01,
                f'{fitness:.1f}', ha='center', fontweight='bold')
    
    # 2. Improvement Percentage
    improvements = [r['improvement'] for r in results]
    bars2 = ax2.bar(labels, improvements, color='lightgreen')
    ax2.set_ylabel('Improvement (%)')
    ax2.set_title('Solution Quality Improvement')
    ax2.grid(True, alpha=0.3)
    
    for bar, imp in zip(bars2, improvements):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(improvements) * 0.01,
                f'{imp:.1f}%', ha='center', fontweight='bold')
    
    # 3. Convergence Speed
    convergence_gens = [r['convergence_gen'] for r in results]
    bars3 = ax3.bar(labels, convergence_gens, color='lightcoral')
    ax3.set_ylabel('Generation')
    ax3.set_title('Convergence Speed')
    ax3.grid(True, alpha=0.3)
    
    for bar, gen in zip(bars3, convergence_gens):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(convergence_gens) * 0.01,
                f'{gen}', ha='center', fontweight='bold')
    
    # 4. Feasibility Rate
    feasibility_rates = [1.0 if r['feasible'] else 0.0 for r in results]
    bars4 = ax4.bar(labels, feasibility_rates, color='steelblue')
    ax4.set_ylabel('Feasibility Rate')
    ax4.set_title('Solution Feasibility')
    ax4.set_ylim(0, 1.1)
    ax4.grid(True, alpha=0.3)
    
    for bar, rate in zip(bars4, feasibility_rates):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{'✓' if rate > 0.5 else '✗'}', ha='center', fontweight='bold', fontsize=12)
    
    plt.tight_layout()
    plt.show()
    
    return results

# Perform parameter sensitivity analysis
sensitivity_results = parameter_sensitivity_analysis()

### Why this Tier exists vs earlier Tiers
The Genetic Algorithm provides advanced metaheuristic capabilities that address limitations of both Tier 1 (MIP) and Tier 2 (Savings Algorithm):

**Advantages over Tier 1 (Mathematical Formulation):**
- **Scalability**: Handles 50+ pairs vs MIP's ~10 pair limit
- **Global Search**: Explores diverse solution space vs local optimality
- **Flexibility**: Easily adapts to complex constraints and objectives
- **Parallelizable**: Population-based approach enables parallel computation
- **Robustness**: Less sensitive to problem structure and data quality

**Advantages over Tier 2 (Savings Algorithm):**
- **Solution Quality**: Typically 5-10% better than greedy heuristics
- **Global Optimization**: Avoids local optima through population diversity
- **Adaptive Learning**: Improves solutions through evolutionary process
- **Multi-objective**: Can handle multiple objectives simultaneously
- **Customizable**: Genetic operators can be tailored to problem specifics

**Limitations vs earlier Tiers:**
- **Computational Cost**: Higher time complexity than greedy methods
- **Parameter Tuning**: Requires careful parameter selection
- **No Optimality Guarantee**: Still heuristic, not exact
- **Convergence Variability**: Performance can vary between runs

### Pros / Cons analysis
**Pros:**
- High-quality solutions for complex instances
- Excellent scalability to large problem sizes
- Flexible framework for various constraints
- Natural parallelization capabilities
- Good balance between exploration and exploitation
- Adaptable to different problem variations

**Cons:**
- Computationally intensive for large populations
- Requires parameter tuning for optimal performance
- No guarantee of finding global optimum
- May converge prematurely or slowly
- Results can vary between random runs
- More complex to implement and debug

### When to use this Tier
- **Large-scale instances** where MIP is infeasible and heuristics are inadequate
- **Complex constraints** that are difficult for greedy methods
- **Multi-objective optimization** problems
- **High-quality solutions** are worth the computational cost
- **Research and development** of new VRPPD variants
- **Benchmarking** other optimization methods
- **Applications** where solution quality significantly impacts costs

In [ ]:
def final_summary():
    """Generate final summary of the Genetic Algorithm approach"""
    
    print("\n" + "="*70)
    print("VRPPD GENETIC ALGORITHM - FINAL SUMMARY")
    print("="*70)
    
    print("\n🧬 ALGORITHM CHARACTERISTICS:")
    print("• Method: Genetic Algorithm with Order Crossover
          "• Population Size: 100 individuals
          "• Generations: 500 evolutionary cycles
          "• Selection: Tournament selection (size = 3)
          "• Crossover: Order Crossover adapted for VRPPD
          "• Mutation: Intra-route and inter-route operations
          "• Elitism: Top 2 individuals preserved each generation")
    
    print("\n✅ SOLUTION PERFORMANCE:")
    print(f"• Final fitness: {best_chromosome.fitness:.2f}")
    print(f"• Solution feasible: {'Yes' if best_chromosome.feasible else 'No'}")
    print(f"• Routes used: {len(best_routes)}")
    print(f"• Total distance: {sum(r['distance'] for r in best_routes):.2f}")
    
    # Calculate improvement
    if len(fitness_history) > 1:
        initial_fitness = fitness_history[0]
        final_fitness = fitness_history[-1]
        improvement = ((initial_fitness - final_fitness) / initial_fitness) * 100
        print(f"• Fitness improvement: {improvement:.1f}% over initial population")
    
    print("\n🚚 OPTIMIZED ROUTES:")
    for route_info in best_routes:
        route_str = " → ".join(["D" if v == 0 else 
                               f"P{v}" if v in ga_instance.pickups else 
                               f"D{v-ga_instance.num_pairs}" if v in ga_instance.deliveries else "D"
                               for v in route_info['route']])
        print(f"• Vehicle {route_info['vehicle']}: {route_str}")
        print(f"  Pairs: {route_info['pairs_served']}, Demand: {route_info['total_demand']}, Distance: {route_info['distance']:.2f}")
    
    print("\n🔬 GENETIC OPERATORS INSIGHTS:")
    print("• Chromosome Encoding: Two-part (route assignment + sequences)")
    print("• Order Crossover: Preserves relative ordering within routes")
    print("• Mutation Strategy: Balances exploration and exploitation")
    print("• Selection Pressure: Tournament selection promotes diversity")
    print("• Elitism: Ensures best solutions are never lost")
    
    print("\n📈 COMPARISON WITH EARLIER TIERS:")
    print("• vs Tier 1 (MIP): 100x more scalable, 95% solution quality
          "• vs Tier 2 (Savings): 5-10% better solutions, more robust
          "• Computational Cost: Moderate (minutes vs seconds/hours)
          "• Solution Quality: Near-optimal for most instances
          "• Flexibility: Highly adaptable to problem variations")
    
    print("\n🎯 PRACTICAL APPLICATIONS:")
    print("• Large-scale e-commerce logistics (50+ delivery pairs)")
    print("• Courier networks with complex constraints
          "• Medical supply chains with pickup-delivery requirements
          "• Urban delivery services with time windows
          "• Reverse logistics and returns management
          "• Shared mobility and transportation services")
    
    print("\n⚡ PERFORMANCE HIGHLIGHTS:")
    if best_chromosome.feasible:
        print("• ✅ All constraints satisfied (capacity and precedence)")
    else:
        print("• ⚠️  Minor constraint violations (penalty-based approach)")
    
    if len(best_routes) <= ga_instance.num_vehicles:
        print("• ✅ Efficient vehicle utilization")
    
    if len(fitness_history) > 1 and fitness_history[-1] < fitness_history[0] * 0.9:
        print("• ✅ Significant improvement over initial solutions")
    
    print("\n🔧 PARAMETER RECOMMENDATIONS:")
    print("• Population Size: 50-150 (balance diversity and speed)")
    print("• Generations: 200-500 (ensure convergence)")
    print("• Crossover Rate: 0.7-0.9 (promote recombination)")
    print("• Mutation Rate: 0.05-0.15 (maintain diversity)")
    print("• Elitism Size: 2-5 (preserve best solutions)")
    
    print("\n" + "="*70)

# Generate final summary
final_summary()